# Linear/Logistic Regression Training on new_features (and new_features + extended)


In [ ]:
import pandas as pd
from pathlib import Path
from datetime import datetime
import warnings

from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression, LinearRegression

from src.utils.training import (
    load_split_info,
    get_task_type_for_dataset_label,
    compute_metrics,
    save_results_incremental,
)
from src.utils.feature_extraction_utils import get_split_info_paths_for_dataset

EXTENDED_FEATURES_DIR = Path('extensive_features')
SPLITS_DIR = EXTENDED_FEATURES_DIR / 'splits'

NEW_FEATURES_DIR = Path('new_features')
RESULTS_ROOT = Path('results')
RESULTS_DIR_NEW = RESULTS_ROOT / 'logreg' / 'new_features'
RESULTS_DIR_NEW_EXT = RESULTS_ROOT / 'logreg' / 'new_features_plus_extended'
RESULTS_DIR_NEW_EXT_BASELINE = RESULTS_ROOT / 'logreg' / 'new_features_plus_extended_shannon_entropies'

SUITE_SUFFIX = '_new'

print("Configuration:")
print(f"  features_dir: {NEW_FEATURES_DIR}")
print(f"  extended_features_dir: {EXTENDED_FEATURES_DIR}")
print(f"  splits: {SPLITS_DIR}")
print(f"  results (new): {RESULTS_DIR_NEW}")
for d in [RESULTS_DIR_NEW, RESULTS_DIR_NEW_EXT, RESULTS_DIR_NEW_EXT_BASELINE]:
    d.mkdir(parents=True, exist_ok=True)
print(f"  results (new+extended): {RESULTS_DIR_NEW_EXT}")


In [ ]:
subfolders = sorted([d.name for d in NEW_FEATURES_DIR.iterdir() if d.is_dir()])
print(f"Found {len(subfolders)} subfolders in default source {NEW_FEATURES_DIR}:")
for sf in subfolders:
    n_files = len(list((NEW_FEATURES_DIR / sf).glob('*_new.csv')))
    print(f"  - {sf}: {n_files} feature files")



In [ ]:
def _sanitize_and_fill(X_train: pd.DataFrame, X_val: pd.DataFrame, X_test: pd.DataFrame):
    """Sanitize column names, replace inf, fill missing values using train statistics."""
    import numpy as np

    def sanitize_column_name(name):
        for char in ['[', ']', '{', '}', '"', "'", ',', ':', ';', '<', '>', '\\', '/']:
            name = str(name).replace(char, '_')
        return name

    sanitized_cols = {col: sanitize_column_name(col) for col in X_train.columns}
    X_train = X_train.rename(columns=sanitized_cols)
    X_val = X_val.rename(columns=sanitized_cols)
    X_test = X_test.rename(columns=sanitized_cols)

    feature_cols = list(X_train.columns)

    if len(feature_cols) > 1:
        drop_col = feature_cols[-1]
        X_train = X_train.drop(columns=[drop_col])
        X_val = X_val.drop(columns=[drop_col])
        X_test = X_test.drop(columns=[drop_col])
        feature_cols = feature_cols[:-1]

    if len(X_train) > 0:
        numeric_cols = X_train.select_dtypes(include=[np.number]).columns
        if len(numeric_cols) > 0:
            X_train[numeric_cols] = X_train[numeric_cols].replace([np.inf, -np.inf], np.nan)
            X_val[numeric_cols] = X_val[numeric_cols].replace([np.inf, -np.inf], np.nan)
            X_test[numeric_cols] = X_test[numeric_cols].replace([np.inf, -np.inf], np.nan)

            train_mean = X_train[numeric_cols].mean().fillna(0)
            X_train[numeric_cols] = X_train[numeric_cols].fillna(train_mean).fillna(0)
            X_val[numeric_cols] = X_val[numeric_cols].fillna(train_mean).fillna(0)
            X_test[numeric_cols] = X_test[numeric_cols].fillna(train_mean).fillna(0)

        non_numeric_cols = X_train.select_dtypes(exclude=[np.number]).columns
        if len(non_numeric_cols) > 0:
            for col in non_numeric_cols:
                mode_value = X_train[col].mode()
                fill_value = mode_value[0] if len(mode_value) > 0 else ''
                X_train[col] = X_train[col].fillna(fill_value)
                X_val[col] = X_val[col].fillna(fill_value)
                X_test[col] = X_test[col].fillna(fill_value)
    else:
        numeric_cols = X_train.select_dtypes(include=[np.number]).columns
        non_numeric_cols = X_train.select_dtypes(exclude=[np.number]).columns
        if len(numeric_cols) > 0:
            X_train[numeric_cols] = X_train[numeric_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
            X_val[numeric_cols] = X_val[numeric_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
            X_test[numeric_cols] = X_test[numeric_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
        if len(non_numeric_cols) > 0:
            X_train[non_numeric_cols] = X_train[non_numeric_cols].fillna('')
            X_val[non_numeric_cols] = X_val[non_numeric_cols].fillna('')
            X_test[non_numeric_cols] = X_test[non_numeric_cols].fillna('')

    return X_train, X_val, X_test, feature_cols


In [ ]:
def _prepare_logreg_data_from_unsplit(features_path: Path,
                                      split_info_path: Path,
                                      splits_dir: Path,
                                      dataset_name: str,
                                      label_column: str):
    """Load unsplit features, apply split_info, merge labels, and return X/y splits."""
    split_info = load_split_info(split_info_path)
    split_id = split_info_path.stem.replace('_split_info', '')

    train_indexes = set(str(x) for x in split_info.get('train', split_info.get('train_indexes', [])))
    val_indexes = set(str(x) for x in split_info.get('val', split_info.get('val_indexes', [])))
    test_indexes = set(str(x) for x in split_info.get('test', split_info.get('test_indexes', [])))

    print(f"Loading features from {features_path.name}...")
    df_full = pd.read_csv(features_path)
    index_col = 'index' if 'index' in df_full.columns else 'Unnamed: 0'
    if index_col not in df_full.columns:
        raise ValueError(f"Index column '{index_col}' not found in features.")
    df_full[index_col] = df_full[index_col].astype(str)

    df_train = df_full[df_full[index_col].isin(train_indexes)].copy()
    df_val = df_full[df_full[index_col].isin(val_indexes)].copy()
    df_test = df_full[df_full[index_col].isin(test_indexes)].copy()

    print(f"  Train: {len(df_train)} samples, Val: {len(df_val)}, Test: {len(df_test)}")

    train_labels_path = splits_dir / f"{split_id}_labels_train.csv"
    val_labels_path = splits_dir / f"{split_id}_labels_val.csv"
    test_labels_path = splits_dir / f"{split_id}_labels_test.csv"

    train_labels = pd.read_csv(train_labels_path)
    val_labels = pd.read_csv(val_labels_path)
    test_labels = pd.read_csv(test_labels_path)
    for labels_df in [train_labels, val_labels, test_labels]:
        labels_df['index'] = labels_df['index'].astype(str)

    train_labels_indexed = train_labels.set_index('index')
    val_labels_indexed = val_labels.set_index('index')
    test_labels_indexed = test_labels.set_index('index')

    df_train = df_train.merge(train_labels_indexed, left_on=index_col, right_index=True, how='left')
    df_val = df_val.merge(val_labels_indexed, left_on=index_col, right_index=True, how='left')
    df_test = df_test.merge(test_labels_indexed, left_on=index_col, right_index=True, how='left')

    if label_column not in df_train.columns:
        raise ValueError(
            f"Label column '{label_column}' not found. Available: "
            f"{[c for c in df_train.columns if c.endswith('_label')]}"
        )

    for split_name, df in [('train', df_train), ('val', df_val), ('test', df_test)]:
        n_nan = df[label_column].isna().sum()
        if n_nan:
            raise ValueError(f"NaN in label '{label_column}' for {split_name}: {n_nan}")

    n_unique_train = df_train[label_column].nunique()
    if n_unique_train < 2:
        raise ValueError(
            f"Label '{label_column}' has only {n_unique_train} unique value(s) in train."
        )

    exclude_cols = {label_column, 'index', 'Unnamed: 0', 'level_0'}
    exclude_cols.update([col for col in df_train.columns if col.startswith('group_')])
    feature_cols = [col for col in df_train.columns if col not in exclude_cols]

    X_train = df_train[feature_cols].copy()
    y_train = pd.Series(df_train[label_column].squeeze())
    X_val = df_val[feature_cols].copy()
    y_val = pd.Series(df_val[label_column].squeeze())
    X_test = df_test[feature_cols].copy()
    y_test = pd.Series(df_test[label_column].squeeze())

    X_train, X_val, X_test, feature_cols = _sanitize_and_fill(X_train, X_val, X_test)

    task_type = get_task_type_for_dataset_label(dataset_name, label_column, y_train)
    label_encoder = None
    if task_type == 'classification' and not pd.api.types.is_numeric_dtype(y_train):
        label_encoder = LabelEncoder()
        label_encoder.fit(pd.concat([y_train, y_val, y_test]))
        y_train = pd.Series(label_encoder.transform(y_train), index=y_train.index)
        y_val = pd.Series(label_encoder.transform(y_val), index=y_val.index)
        y_test = pd.Series(label_encoder.transform(y_test), index=y_test.index)
        print(f"Encoded {len(label_encoder.classes_)} classes")

    return (
        X_train, y_train,
        X_val, y_val,
        X_test, y_test,
        feature_cols,
        task_type,
        label_encoder,
        split_info,
        split_id,
    )


In [ ]:
def _prepare_logreg_data_unsplit_plus_extended(features_path: Path,
                                               split_info_path: Path,
                                               splits_dir: Path,
                                               extended_features_dir: Path,
                                               dataset_name: str,
                                               label_column: str,
                                               include_shannon_entropies: bool = True):
    """Like _prepare_logreg_data_from_unsplit, but concatenates extended_features."""
    split_info = load_split_info(split_info_path)
    split_id = split_info_path.stem.replace('_split_info', '')

    train_indexes = set(str(x) for x in split_info.get('train', split_info.get('train_indexes', [])))
    val_indexes = set(str(x) for x in split_info.get('val', split_info.get('val_indexes', [])))
    test_indexes = set(str(x) for x in split_info.get('test', split_info.get('test_indexes', [])))

    print(f"Loading new features from {features_path.name}...")
    df_full = pd.read_csv(features_path)
    index_col = 'index' if 'index' in df_full.columns else 'Unnamed: 0'
    if index_col not in df_full.columns:
        raise ValueError(f"Index column '{index_col}' not found in features.")
    df_full[index_col] = df_full[index_col].astype(str)

    df_train_new = df_full[df_full[index_col].isin(train_indexes)].copy()
    df_val_new = df_full[df_full[index_col].isin(val_indexes)].copy()
    df_test_new = df_full[df_full[index_col].isin(test_indexes)].copy()

    ext_train_path = extended_features_dir / f"{split_id}_extended_features_train.csv"
    ext_val_path = extended_features_dir / f"{split_id}_extended_features_val.csv"
    ext_test_path = extended_features_dir / f"{split_id}_extended_features_test.csv"

    ext_train = pd.read_csv(ext_train_path)
    ext_val = pd.read_csv(ext_val_path)
    ext_test = pd.read_csv(ext_test_path)
    for df in [ext_train, ext_val, ext_test]:
        df['index'] = df['index'].astype(str)

    df_train = df_train_new.merge(ext_train, on='index', how='inner', suffixes=('', '_ext'))
    df_val = df_val_new.merge(ext_val, on='index', how='inner', suffixes=('', '_ext'))
    df_test = df_test_new.merge(ext_test, on='index', how='inner', suffixes=('', '_ext'))

    if include_shannon_entropies:
        suite_suffix = '_new'
        dataset_base = features_path.stem[:-len(suite_suffix)] if features_path.stem.endswith(suite_suffix) else features_path.stem
        baseline_path = NEW_FEATURES_DIR / 'shannon_entropies' / f"{dataset_base}{suite_suffix}.csv"
        baseline_df = pd.read_csv(baseline_path)
        baseline_idx_col = 'index' if 'index' in baseline_df.columns else 'Unnamed: 0'
        baseline_df[baseline_idx_col] = baseline_df[baseline_idx_col].astype(str)
        baseline_tr = baseline_df[baseline_df[baseline_idx_col].isin(train_indexes)].copy()
        baseline_va = baseline_df[baseline_df[baseline_idx_col].isin(val_indexes)].copy()
        baseline_te = baseline_df[baseline_df[baseline_idx_col].isin(test_indexes)].copy()
        baseline_tr = baseline_tr.rename(columns={baseline_idx_col: 'index'})
        baseline_va = baseline_va.rename(columns={baseline_idx_col: 'index'})
        baseline_te = baseline_te.rename(columns={baseline_idx_col: 'index'})
        df_train = df_train.merge(baseline_tr, on='index', how='inner', suffixes=('', '_bse'))
        df_val = df_val.merge(baseline_va, on='index', how='inner', suffixes=('', '_bse'))
        df_test = df_test.merge(baseline_te, on='index', how='inner', suffixes=('', '_bse'))
        print(f"  Train: {len(df_train)} samples (new+extended+shannon_entropies), Val: {len(df_val)}, Test: {len(df_test)}")
    else:
        print(f"  Train: {len(df_train)} samples (new+extended), Val: {len(df_val)}, Test: {len(df_test)}")

    train_labels_path = splits_dir / f"{split_id}_labels_train.csv"
    val_labels_path = splits_dir / f"{split_id}_labels_val.csv"
    test_labels_path = splits_dir / f"{split_id}_labels_test.csv"

    train_labels = pd.read_csv(train_labels_path)
    val_labels = pd.read_csv(val_labels_path)
    test_labels = pd.read_csv(test_labels_path)
    for labels_df in [train_labels, val_labels, test_labels]:
        labels_df['index'] = labels_df['index'].astype(str)

    train_labels_indexed = train_labels.set_index('index')
    val_labels_indexed = val_labels.set_index('index')
    test_labels_indexed = test_labels.set_index('index')

    df_train = df_train.merge(train_labels_indexed, left_on=index_col, right_index=True, how='left')
    df_val = df_val.merge(val_labels_indexed, left_on=index_col, right_index=True, how='left')
    df_test = df_test.merge(test_labels_indexed, left_on=index_col, right_index=True, how='left')

    if label_column not in df_train.columns:
        raise ValueError(
            f"Label column '{label_column}' not found. Available: "
            f"{[c for c in df_train.columns if c.endswith('_label')]}"
        )

    for split_name, df in [('train', df_train), ('val', df_val), ('test', df_test)]:
        n_nan = df[label_column].isna().sum()
        if n_nan:
            raise ValueError(f"NaN in label '{label_column}' for {split_name}: {n_nan}")

    n_unique_train = df_train[label_column].nunique()
    if n_unique_train < 2:
        raise ValueError(
            f"Label '{label_column}' has only {n_unique_train} unique value(s) in train."
        )

    exclude_cols = {label_column, 'index', 'Unnamed: 0', 'level_0'}
    exclude_cols.update([col for col in df_train.columns if col.startswith('group_')])
    feature_cols = [col for col in df_train.columns if col not in exclude_cols]

    X_train = df_train[feature_cols].copy()
    y_train = pd.Series(df_train[label_column].squeeze())
    X_val = df_val[feature_cols].copy()
    y_val = pd.Series(df_val[label_column].squeeze())
    X_test = df_test[feature_cols].copy()
    y_test = pd.Series(df_test[label_column].squeeze())

    X_train, X_val, X_test, feature_cols = _sanitize_and_fill(X_train, X_val, X_test)

    task_type = get_task_type_for_dataset_label(dataset_name, label_column, y_train)
    label_encoder = None
    if task_type == 'classification' and not pd.api.types.is_numeric_dtype(y_train):
        label_encoder = LabelEncoder()
        label_encoder.fit(pd.concat([y_train, y_val, y_test]))
        y_train = pd.Series(label_encoder.transform(y_train), index=y_train.index)
        y_val = pd.Series(label_encoder.transform(y_val), index=y_val.index)
        y_test = pd.Series(label_encoder.transform(y_test), index=y_test.index)
        print(f"Encoded {len(label_encoder.classes_)} classes")

    return (
        X_train, y_train,
        X_val, y_val,
        X_test, y_test,
        feature_cols,
        task_type,
        label_encoder,
        split_info,
        split_id,
    )


In [ ]:
def run_linear_or_logistic_trainval(X_train, y_train, X_val, y_val, X_test, y_test,
                                   task_type: str):
    """Train on train+val concatenated, evaluate on test.

    - classification -> LogisticRegression
    - regression -> LinearRegression
    """
    X_trainval = pd.concat([X_train, X_val], ignore_index=True)
    y_trainval = pd.concat([y_train, y_val], ignore_index=True)

    if task_type == 'classification':
        model = LogisticRegression(
            max_iter=1000,
            multi_class='auto',
        )
    else:
        model = LinearRegression()

    model.fit(X_trainval, y_trainval)

    y_trainval_pred = model.predict(X_trainval)
    y_test_pred = model.predict(X_test)

    train_metrics = compute_metrics(y_trainval, y_trainval_pred, task_type)
    test_metrics = compute_metrics(y_test, y_test_pred, task_type)

    training_info = {
        'task_type': task_type,
        'model_family': 'LogisticRegression' if task_type == 'classification' else 'LinearRegression',
        'n_features': X_trainval.shape[1],
        'n_trainval_samples': len(X_trainval),
        'n_train_samples': len(X_train),
        'n_val_samples': len(X_val),
    }

    return model, training_info, train_metrics, test_metrics, {
        'y_trainval': y_trainval.values,
        'y_trainval_pred': y_trainval_pred,
        'y_test': y_test.values,
        'y_test_pred': y_test_pred,
    }


In [ ]:
def train_logreg_new_features_subfolder(subfolder: str, skip_existing: bool = True):
    """Train Logistic Regression for all datasets in a new_features subfolder (new_features only)."""
    subfolder_path = NEW_FEATURES_DIR / subfolder
    results_file = RESULTS_DIR_NEW / f'logreg_results_{subfolder}.csv'
    existing_keys = set()

    if results_file.exists() and skip_existing:
        existing_df = pd.read_csv(results_file)
        if all(c in existing_df.columns for c in ['dataset', 'label', 'feature_battery']):
            existing_keys = set(zip(
                existing_df['dataset'].astype(str),
                existing_df['label'].astype(str),
                existing_df['feature_battery'].astype(str),
            ))

    feature_files = list(subfolder_path.glob('*_new.csv'))
    all_results = []

    for feat_path in feature_files:
        stem = feat_path.stem
        if not stem.endswith(SUITE_SUFFIX):
            continue
        dataset_base = stem[:-len(SUITE_SUFFIX)]

        split_paths = get_split_info_paths_for_dataset(SPLITS_DIR, dataset_base)
        if not split_paths:
            continue

        print(f"  Processing {feat_path.name} -> {dataset_base} ({len(split_paths)} splits)")

        for split_path in split_paths:
            split_id = split_path.stem.replace('_split_info', '')

            si = load_split_info(split_path)
            label_col = si.get('label_col')
            if not label_col:
                continue

            key = (split_id, label_col, subfolder)
            if key in existing_keys:
                continue

            try:
                (
                    X_train, y_train,
                    X_val, y_val,
                    X_test, y_test,
                    feature_cols,
                    task_type,
                    label_encoder,
                    split_info,
                    split_id_actual,
                ) = _prepare_logreg_data_from_unsplit(
                    features_path=feat_path,
                    split_info_path=split_path,
                    splits_dir=SPLITS_DIR,
                    dataset_name=dataset_base,
                    label_column=label_col,
                )

                model, training_info, train_metrics, test_metrics, predictions = run_linear_or_logistic_trainval(
                    X_train, y_train, X_val, y_val, X_test, y_test, task_type
                )

                result_dict = {
                    'dataset': split_id,
                    'label': label_col,
                    'feature_battery': subfolder,
                    'task_type': task_type,
                    'best_model': 'LogisticRegression' if task_type == 'classification' else 'LinearRegression',
                    'n_features': len(feature_cols),
                    'n_train': len(X_train) + len(X_val),
                    'n_val': 0,
                    'n_test': len(X_test),
                    'timestamp': datetime.now().isoformat(),
                }
                for k, v in train_metrics.items():
                    if isinstance(v, (int, float, str)):
                        result_dict[f'train_{k}'] = v
                for k, v in test_metrics.items():
                    if isinstance(v, (int, float, str)):
                        result_dict[f'test_{k}'] = v

                all_results.append(result_dict)
                save_results_incremental(result_dict, results_file)
                print(f"      Saved: {split_id} / {label_col}")
                print(f"      Test F1: {result_dict.get('test_f1', 'N/A')}")

            except Exception as e:
                result_dict = {
                    'dataset': split_id,
                    'label': label_col,
                    'feature_battery': subfolder,
                    'error': str(e),
                    'timestamp': datetime.now().isoformat(),
                }
                all_results.append(result_dict)
                save_results_incremental(result_dict, results_file)
                print(f"      Error: {e}")

    return all_results


In [ ]:


def train_logreg_new_features_plus_extended(subfolder: str, skip_existing: bool = True):
    """Train Logistic Regression for all datasets in a new_features subfolder (new_features + extended)."""
    subfolder_path = NEW_FEATURES_DIR / subfolder
    results_file = RESULTS_DIR_NEW_EXT / f'logreg_results_{subfolder}.csv'
    existing_keys = set()

    feature_battery = f'{subfolder}+extended'

    if results_file.exists() and skip_existing:
        existing_df = pd.read_csv(results_file)
        if all(c in existing_df.columns for c in ['dataset', 'label', 'feature_battery']):
            existing_keys = set(zip(
                existing_df['dataset'].astype(str),
                existing_df['label'].astype(str),
                existing_df['feature_battery'].astype(str),
            ))

    feature_files = list(subfolder_path.glob('*_new.csv'))
    all_results = []

    for feat_path in feature_files:
        stem = feat_path.stem
        if not stem.endswith(SUITE_SUFFIX):
            continue
        dataset_base = stem[:-len(SUITE_SUFFIX)]

        split_paths = get_split_info_paths_for_dataset(SPLITS_DIR, dataset_base)
        if not split_paths:
            continue

        print(f"  Processing {feat_path.name} -> {dataset_base} ({len(split_paths)} splits)")

        for split_path in split_paths:
            split_id = split_path.stem.replace('_split_info', '')

            si = load_split_info(split_path)
            label_col = si.get('label_col')
            if not label_col:
                continue

            key = (split_id, label_col, feature_battery)
            if key in existing_keys:
                continue

            try:
                (
                    X_train, y_train,
                    X_val, y_val,
                    X_test, y_test,
                    feature_cols,
                    task_type,
                    label_encoder,
                    split_info,
                    split_id_actual,
                ) = _prepare_logreg_data_unsplit_plus_extended(
                    features_path=feat_path,
                    split_info_path=split_path,
                    splits_dir=SPLITS_DIR,
                    extended_features_dir=EXTENDED_FEATURES_DIR,
                    dataset_name=dataset_base,
                    label_column=label_col,
                )

                model, training_info, train_metrics, test_metrics, predictions = run_linear_or_logistic_trainval(
                    X_train, y_train, X_val, y_val, X_test, y_test, task_type
                )

                result_dict = {
                    'dataset': split_id,
                    'label': label_col,
                    'feature_battery': feature_battery,
                    'task_type': task_type,
                    'best_model': 'LogisticRegression' if task_type == 'classification' else 'LinearRegression',
                    'n_features': len(feature_cols),
                    'n_train': len(X_train) + len(X_val),
                    'n_val': 0,
                    'n_test': len(X_test),
                    'timestamp': datetime.now().isoformat(),
                }
                for k, v in train_metrics.items():
                    if isinstance(v, (int, float, str)):
                        result_dict[f'train_{k}'] = v
                for k, v in test_metrics.items():
                    if isinstance(v, (int, float, str)):
                        result_dict[f'test_{k}'] = v

                all_results.append(result_dict)
                save_results_incremental(result_dict, results_file)
                print(f"      Saved: {split_id} / {label_col}")
                print(f"      Test F1: {result_dict.get('test_f1', 'N/A')}")

            except Exception as e:
                result_dict = {
                    'dataset': split_id,
                    'label': label_col,
                    'feature_battery': feature_battery,
                    'error': str(e),
                    'timestamp': datetime.now().isoformat(),
                }
                all_results.append(result_dict)
                save_results_incremental(result_dict, results_file)
                print(f"      Error: {e}")

    return all_results


def train_logreg_new_features_plus_extended_baseline(subfolder: str, skip_existing: bool = True):
    if subfolder == 'shannon_entropies':
        print("Skipping shannon_entropies for +extended+shannon_entropies to avoid duplicate battery")
        return

    """Train Logistic Regression for all datasets in a new_features subfolder (new_features + extended)."""
    subfolder_path = NEW_FEATURES_DIR / subfolder
    results_file = RESULTS_DIR_NEW_EXT_BASELINE / f'logreg_results_{subfolder}.csv'
    existing_keys = set()

    feature_battery = f'{subfolder}+extended+shannon_entropies'

    if results_file.exists() and skip_existing:
        existing_df = pd.read_csv(results_file)
        if all(c in existing_df.columns for c in ['dataset', 'label', 'feature_battery']):
            existing_keys = set(zip(
                existing_df['dataset'].astype(str),
                existing_df['label'].astype(str),
                existing_df['feature_battery'].astype(str),
            ))

    feature_files = list(subfolder_path.glob('*_new.csv'))
    all_results = []

    for feat_path in feature_files:
        stem = feat_path.stem
        if not stem.endswith(SUITE_SUFFIX):
            continue
        dataset_base = stem[:-len(SUITE_SUFFIX)]

        split_paths = get_split_info_paths_for_dataset(SPLITS_DIR, dataset_base)
        if not split_paths:
            continue

        print(f"  Processing {feat_path.name} -> {dataset_base} ({len(split_paths)} splits)")

        for split_path in split_paths:
            split_id = split_path.stem.replace('_split_info', '')

            si = load_split_info(split_path)
            label_col = si.get('label_col')
            if not label_col:
                continue

            key = (split_id, label_col, feature_battery)
            if key in existing_keys:
                continue

            try:
                (
                    X_train, y_train,
                    X_val, y_val,
                    X_test, y_test,
                    feature_cols,
                    task_type,
                    label_encoder,
                    split_info,
                    split_id_actual,
                ) = _prepare_logreg_data_unsplit_plus_extended(
                    features_path=feat_path,
                    split_info_path=split_path,
                    splits_dir=SPLITS_DIR,
                    extended_features_dir=EXTENDED_FEATURES_DIR,
                    dataset_name=dataset_base,
                    label_column=label_col,
                )

                model, training_info, train_metrics, test_metrics, predictions = run_linear_or_logistic_trainval(
                    X_train, y_train, X_val, y_val, X_test, y_test, task_type
                )

                result_dict = {
                    'dataset': split_id,
                    'label': label_col,
                    'feature_battery': feature_battery,
                    'task_type': task_type,
                    'best_model': 'LogisticRegression' if task_type == 'classification' else 'LinearRegression',
                    'n_features': len(feature_cols),
                    'n_train': len(X_train) + len(X_val),
                    'n_val': 0,
                    'n_test': len(X_test),
                    'timestamp': datetime.now().isoformat(),
                }
                for k, v in train_metrics.items():
                    if isinstance(v, (int, float, str)):
                        result_dict[f'train_{k}'] = v
                for k, v in test_metrics.items():
                    if isinstance(v, (int, float, str)):
                        result_dict[f'test_{k}'] = v

                all_results.append(result_dict)
                save_results_incremental(result_dict, results_file)
                print(f"      Saved: {split_id} / {label_col}")
                print(f"      Test F1: {result_dict.get('test_f1', 'N/A')}")

            except Exception as e:
                result_dict = {
                    'dataset': split_id,
                    'label': label_col,
                    'feature_battery': feature_battery,
                    'error': str(e),
                    'timestamp': datetime.now().isoformat(),
                }
                all_results.append(result_dict)
                save_results_incremental(result_dict, results_file)
                print(f"      Error: {e}")

    return all_results




In [ ]:
from src.utils.training import print_results_summary




RESULTS_DIR_EXT_ONLY = RESULTS_ROOT / 'logreg' / 'extended_only'
RESULTS_DIR_EXT_ONLY.mkdir(parents=True, exist_ok=True)

def _prepare_extended_only(split_id: str, label_col: str):
    train_path = EXTENDED_FEATURES_DIR / f"{split_id}_extended_features_train.csv"
    val_path = EXTENDED_FEATURES_DIR / f"{split_id}_extended_features_val.csv"
    test_path = EXTENDED_FEATURES_DIR / f"{split_id}_extended_features_test.csv"

    df_train = pd.read_csv(train_path)
    df_val = pd.read_csv(val_path)
    df_test = pd.read_csv(test_path)

    index_col = 'index' if 'index' in df_train.columns else 'Unnamed: 0'
    for df in (df_train, df_val, df_test):
        if index_col not in df.columns:
            raise ValueError(f"Index column '{index_col}' not found in extended features")
        df[index_col] = df[index_col].astype(str)

    train_labels = pd.read_csv(SPLITS_DIR / f"{split_id}_labels_train.csv")
    val_labels = pd.read_csv(SPLITS_DIR / f"{split_id}_labels_val.csv")
    test_labels = pd.read_csv(SPLITS_DIR / f"{split_id}_labels_test.csv")
    for ldf in (train_labels, val_labels, test_labels):
        ldf['index'] = ldf['index'].astype(str)

    df_train = df_train.merge(train_labels.set_index('index'), left_on=index_col, right_index=True, how='left')
    df_val = df_val.merge(val_labels.set_index('index'), left_on=index_col, right_index=True, how='left')
    df_test = df_test.merge(test_labels.set_index('index'), left_on=index_col, right_index=True, how='left')

    if label_col not in df_train.columns:
        raise ValueError(f"Label '{label_col}' not found after merge")

    exclude_cols = {label_col, 'index', 'Unnamed: 0', 'level_0'}
    exclude_cols.update([c for c in df_train.columns if c.startswith('group_')])
    feature_cols = [c for c in df_train.columns if c not in exclude_cols]

    X_train = df_train[feature_cols].copy()
    y_train = pd.Series(df_train[label_col].squeeze())
    X_val = df_val[feature_cols].copy()
    y_val = pd.Series(df_val[label_col].squeeze())
    X_test = df_test[feature_cols].copy()
    y_test = pd.Series(df_test[label_col].squeeze())

    X_train, X_val, X_test, feature_cols = _sanitize_and_fill(X_train, X_val, X_test)

    task_type = get_task_type_for_dataset_label(split_id, label_col, y_train)
    if task_type == 'classification' and not pd.api.types.is_numeric_dtype(y_train):
        le = LabelEncoder()
        le.fit(pd.concat([y_train, y_val, y_test]))
        y_train = pd.Series(le.transform(y_train), index=y_train.index)
        y_val = pd.Series(le.transform(y_val), index=y_val.index)
        y_test = pd.Series(le.transform(y_test), index=y_test.index)

    return X_train, y_train, X_val, y_val, X_test, y_test, feature_cols, task_type


def train_logreg_extended_only(skip_existing: bool = True):
    results_file = RESULTS_DIR_EXT_ONLY / 'logreg_results_extended_only.csv'
    existing_keys = set()
    if results_file.exists() and skip_existing:
        df_exist = pd.read_csv(results_file)
        if all(c in df_exist.columns for c in ['dataset', 'label', 'feature_battery']):
            existing_keys = set(zip(df_exist['dataset'].astype(str), df_exist['label'].astype(str), df_exist['feature_battery'].astype(str)))

    split_paths = sorted(SPLITS_DIR.glob('*_split_info.json'))
    print(f"[extended_only] Found {len(split_paths)} split_info files in {SPLITS_DIR}")
    for sp in split_paths:
        split_id = sp.stem.replace('_split_info', '')

        si = load_split_info(sp)
        label_col = si.get('label_col')
        if not label_col:
            continue

        key = (split_id, label_col, 'extended_only')
        if key in existing_keys:
            continue

        try:
            X_train, y_train, X_val, y_val, X_test, y_test, feature_cols, task_type = _prepare_extended_only(split_id, label_col)
            model, training_info, train_metrics, test_metrics, _ = run_linear_or_logistic_trainval(
                X_train, y_train, X_val, y_val, X_test, y_test, task_type
            )

            result_dict = {
                'dataset': split_id,
                'label': label_col,
                'feature_battery': 'extended_only',
                'task_type': task_type,
                'best_model': 'LogisticRegression' if task_type == 'classification' else 'LinearRegression',
                'n_features': len(feature_cols),
                'n_train': len(X_train) + len(X_val),
                'n_val': 0,
                'n_test': len(X_test),
                'timestamp': datetime.now().isoformat(),
            }
            for k, v in train_metrics.items():
                if isinstance(v, (int, float, str)):
                    result_dict[f'train_{k}'] = v
            for k, v in test_metrics.items():
                if isinstance(v, (int, float, str)):
                    result_dict[f'test_{k}'] = v

            save_results_incremental(result_dict, results_file)
            print(f"  [extended_only] Saved: {split_id} / {label_col} (Test F1={result_dict.get('test_f1','N/A')})")

        except Exception as e:
            save_results_incremental({
                'dataset': split_id,
                'label': label_col,
                'feature_battery': 'extended_only',
                'error': str(e),
                'timestamp': datetime.now().isoformat(),
            }, results_file)
            print(f"  [extended_only] Error: {split_id} / {label_col}: {e}")


print("\n" + "#" * 80)
print("LogReg — extended_features only (train+val)")
print("#" * 80)
train_logreg_extended_only(skip_existing=True)
ext_path = RESULTS_DIR_EXT_ONLY / 'logreg_results_extended_only.csv'
if ext_path.exists():
    print_results_summary(pd.read_csv(ext_path))



print("\n" + "#" * 80)
print(f"Features root: {NEW_FEATURES_DIR}")
print(f"Results (new): {RESULTS_DIR_NEW}")
print(f"Results (new+extended): {RESULTS_DIR_NEW_EXT}")
print("#" * 80)

subfolders = sorted([d.name for d in NEW_FEATURES_DIR.iterdir() if d.is_dir()])
print(f"Found {len(subfolders)} subfolders in {NEW_FEATURES_DIR}")


to_run = subfolders
if not to_run:
    print("WARNING: Nothing to run (to_run is empty).")

for subfolder in to_run:
    print("\n" + "=" * 80)
    print(f"LogReg (new_features only, train+val) — subfolder: {subfolder}")
    print("=" * 80)
    train_logreg_new_features_subfolder(subfolder, skip_existing=True)
    results_path = RESULTS_DIR_NEW / f'logreg_results_{subfolder}.csv'
    if results_path.exists():
        df = pd.read_csv(results_path)
        print_results_summary(df)

    print("\n" + "=" * 80)
    print("4) LOGREG (new_features + extended + shannon_entropies) — subfolder: " + str(subfolder))
    print("=" * 80)
    train_logreg_new_features_plus_extended_baseline(subfolder, skip_existing=True)
    results_path = RESULTS_DIR_NEW_EXT_BASELINE / f'logreg_results_{subfolder}.csv'
    if results_path.exists():
        df = pd.read_csv(results_path)
        print_results_summary(df)

    print("\n" + "=" * 80)
    print("4) LOGREG (new_features + extended + shannon_entropies) — subfolder: " + str(subfolder))
    print("=" * 80)
    train_logreg_new_features_plus_extended_baseline(subfolder, skip_existing=True)
    results_path = RESULTS_DIR_NEW_EXT_BASELINE / f'logreg_results_{subfolder}.csv'
    if results_path.exists():
        df = pd.read_csv(results_path)
        print_results_summary(df)

    print("\n" + "=" * 80)
    print(f"LogReg (new_features + extended_features, train+val) — subfolder: {subfolder}")
    print("=" * 80)
    train_logreg_new_features_plus_extended(subfolder, skip_existing=True)
    results_path = RESULTS_DIR_NEW_EXT / f'logreg_results_{subfolder}.csv'
    if results_path.exists():
        df = pd.read_csv(results_path)
        print_results_summary(df)
